# Sentiment Analysis Using TF-IDF (Term Frequency-Inverse Document Frequency) and XGBoost

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Term Frequency-Inverse Document Frequency (TfidfVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.1 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the TF-IDF (Term Frequency-Inverse Document Frequency) model**

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
tf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2)) # max_features=10000, ngram_range=(1,2)
X_train = tf.fit_transform(X_train)
X_test = tf.transform(X_test)

In [15]:
print(X_train.shape)        # (n_samples, n_features)
print(X_train.shape[0])     # number of training samples
print(X_train.shape[1])     # number of features (vocabulary size)

(19923, 10000)
19923
10000


In [16]:
print(X_test.shape)        # (n_samples, n_features)
print(X_test.shape[0])     # number of training samples
print(X_test.shape[1])     # number of features (vocabulary size)

(4981, 10000)
4981
10000


# Tuning Hyperparameters with HalvingRandomSearchCV

In [17]:
from xgboost import XGBClassifier
from sklearn.experimental import enable_halving_search_cv  # Required
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import uniform, randint

# Initialize XGBoost
xgb = XGBClassifier(
    random_state=0,
    eval_metric='logloss'
)

# Define parameter grid
param_grid = {
    "learning_rate": uniform(0.01, 0.19),      # samples continuously from 0.01 to 0.20
    "max_depth": randint(3, 8),                 # samples integers from 3 to 7
    "min_child_weight": randint(1, 6),          # samples integers from 1 to 5
    "subsample": uniform(0.7, 0.3),             # samples from 0.7 to 1.0
    "colsample_bytree": uniform(0.7, 0.3)       # samples from 0.7 to 1.0
}

# Halving Random Search
halving_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    factor=3,                # Controls how aggressively candidates are reduced
    resource='n_estimators',  # Progressively increase number of boosting rounds
    min_resources=20,         # Starting number of trees for early elimination rounds
    max_resources=500,        # Final round trains up to 500 trees
    scoring='accuracy',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Fit on training data
halving_search.fit(X_train, y_train)

# Best parameters and score
print("Best Parameters:", halving_search.best_params_)
print("Best CV Score:", halving_search.best_score_)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 20
max_resources_: 500
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 25
n_resources: 20
Fitting 3 folds for each of 25 candidates, totalling 75 fits
----------
iter: 1
n_candidates: 9
n_resources: 60
Fitting 3 folds for each of 9 candidates, totalling 27 fits
----------
iter: 2
n_candidates: 3
n_resources: 180
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best Parameters: {'colsample_bytree': np.float64(0.8525712073494107), 'learning_rate': np.float64(0.18243763004595767), 'max_depth': 7, 'min_child_weight': 3, 'subsample': np.float64(0.7609183674204307), 'n_estimators': 180}
Best CV Score: 0.8577523465341566
[CV] END colsample_bytree=0.8123620356542087, learning_rate=0.19063571821788408, max_depth=5, min_child_weight=5, n_estimators=20, subsample=0.879055047383946; total time=  12.2s
[CV] END colsample_bytree=0.9124217733388136, learning_rate=0.013911053916202464, 

# **Training the XGBoost model on the Training set**

In [17]:
from xgboost import XGBClassifier
classifier = XGBClassifier(booster='gbtree', eval_metric='logloss', subsample=0.7609183674204307, n_estimators=1000, min_child_weight=3, max_depth=7, learning_rate=0.18243763004595767, colsample_bytree=0.8525712073494107, random_state = 0)  
classifier.fit(X_train, y_train)

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8525712073494107, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.18243763004595767,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

# **Predicting the Test set results**

In [18]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


# **Evaluating the Model Performance**

In [19]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2174  346]
 [ 295 2166]]

Accuracy Score:
0.8713109817305762

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      2520
           1       0.86      0.88      0.87      2461

    accuracy                           0.87      4981
   macro avg       0.87      0.87      0.87      4981
weighted avg       0.87      0.87      0.87      4981


ROC-AUC Score:
0.8714142205710673

Mean Accuracy:
0.8623199317371881

Standard Deviation:
0.0028894921376548563


# XGBoost Performance Analysis for Text Classification (TF-IDF)

# 1. Objective

The objective of this experiment was to evaluate the effectiveness of **XGBoost** for binary text classification using features generated by **TfidfVectorizer**, and to compare its performance against the other previously evaluated TF-IDF classifiers.

The experiment aims to answer the following research questions:

* How does XGBoost perform on TF-IDF-weighted features compared to an untuned baseline?
* Does increasing `n_estimators` beyond what the search recommended improve results further?
* Does hyperparameter tuning found at one vocabulary size transfer well to larger vocabulary sizes?
* How does vocabulary size affect performance?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Feature Extraction

| Configuration | Parameters                                     |
| -------------- | ------------------------------------------------ |
| Baseline       | `TfidfVectorizer()` (default settings)            |
| C1             | max_features=5000, ngram_range=(1,2)              |
| C2             | max_features=10000, ngram_range=(1,2)             |
| C3             | max_features=15000, ngram_range=(1,2)             |
| C4             | max_features=20000, min_df=2, ngram_range=(1,2)   |

Only the `gbtree` booster was evaluated in this experiment.

---

# 3. Hyperparameter Optimization Strategy

`HalvingRandomSearchCV` was run **once, at the 5,000-feature configuration**, using `resource='n_estimators'` with `max_resources=500` and a continuous parameter distribution:

```python
param_grid = {
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(3, 8),
    "min_child_weight": randint(1, 6),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3)
}
```

The search selected `n_estimators=180`, which was evaluated exactly as found. A follow-up manual test then raised `n_estimators` to **1000 — beyond the search's own `max_resources=500` ceiling** — while keeping every other parameter fixed, and found a further improvement (Section 6). That 1000-estimator configuration was then carried forward and reused directly at 10K, 15K, and 20K features **without re-running the search at those larger vocabulary sizes**. This is a reasonable practical shortcut, but it means the 10K/15K/20K results reflect a configuration optimized for (and validated at) 5K features, not independently searched for at their own scale.

---

# 4. Hyperparameters Explored and Verification

### Best Parameters Found by the Search (5K features)

```python
{'colsample_bytree': 0.8526, 'learning_rate': 0.1824, 'max_depth': 7,
 'min_child_weight': 3, 'subsample': 0.7609, 'n_estimators': 180}
```
Best CV Score reported by the search: **0.8570**

### Verification

| Step | n_estimators | Matches search? |
| ---- | ------------- | ----------------- |
| Initial evaluation (5K)         | 180  | Exact match |
| Manual follow-up test (5K)      | 1000 | Deliberate deviation, not a mismatch — see Section 6 |
| Carried forward to 10K/15K/20K  | 1000 | Reuses the manual follow-up value, not the search's original recommendation |

---

# 5. Experimental Results

| Configuration                       | Accuracy   | ROC-AUC    | CV Mean | CV Std |
| -------------------------------------- | ---------- | ---------- | ------- | ------ |
| Baseline (default, no tune)            | 84.62%     | 0.8465     | 84.24%  | 0.0013 |
| 5K, default (untuned)                   | 86.29%     | 0.8631     | 84.95%  | 0.0025 |
| 5K, tuned (n_estimators=180, as found)  | 86.27%     | 0.8629     | 85.70%  | 0.0023 |
| 5K, tuned (n_estimators=1000, manual)   | 86.95%     | 0.8697     | 85.98%  | 0.0020 |
| **10K, tuned (n_estimators=1000)**      | **87.13%** | **0.8714** | 86.23%  | 0.0029 |
| 15K, tuned (n_estimators=1000)          | 87.07%     | 0.8709     | 86.29%  | 0.0024 |
| 20K (min_df=2), tuned (n_estimators=1000) | 86.85%   | 0.8687     | 86.40%  | 0.0016 |

---

# 6. Performance Analysis

## Effect of Increasing n_estimators Beyond the Search's Recommendation

| n_estimators | Accuracy | ROC-AUC |
| ------------- | -------- | ------- |
| 180 (search-selected) | 86.27% | 0.8629 |
| 1000 (manual)          | 86.95% | 0.8697 |

Raising `n_estimators` from 180 to 1000 improved accuracy by **0.68 points**, even though the search's own `max_resources=500` ceiling never let it explore beyond 500 trees. This suggests the halving search's resource budget was set conservatively relative to what this model could actually benefit from — with `learning_rate≈0.18`, the model apparently continued improving well past the number of trees the search was allowed to try. Worth considering a higher `max_resources` in future searches for this model.

## Effect of Hyperparameter Optimization

Comparing the 5K default (86.29%) to the 5K tuned runs, the 180-tree tuned model (86.27%) is essentially a wash against the default, while the 1000-tree version (86.95%) is a modest but real improvement — the gain from tuning here is much more modest than what was seen in the earlier CountVectorizer XGBoost experiment, and depended specifically on going beyond the search's own recommendation.

## Effect of Vocabulary Size

| Features | Accuracy   |
| -------- | ---------- |
| 5K       | 86.95%     |
| **10K**  | **87.13%** |
| 15K      | 87.07%     |
| 20K      | 86.85%     |

Accuracy peaks at 10K and gently declines through 15K and 20K — the same "rise then plateau/decline around 10K" pattern seen in the Decision Tree and Random Forest TF-IDF experiments. Since 10K-20K all reuse hyperparameters validated only at 5K (Section 3), it's possible a fresh search at 15K or 20K would shift this picture, but the consistency with other models' vocabulary-size patterns suggests this is a genuine property of the dataset rather than purely a tuning artifact.

---

# 7. Precision and Recall Analysis

### Baseline (untuned, default TF-IDF)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.87      | 0.82   | 0.84     |
| 1     | 0.83      | 0.87   | 0.85     |

### Best Model — 10K Features (Tuned)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.88      | 0.86   | 0.87     |
| 1     | 0.86      | 0.88   | 0.87     |

### 15K Features (Tuned)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.89      | 0.85   | 0.87     |
| 1     | 0.86      | 0.89   | 0.87     |

The best model shows the most balanced precision/recall of the TF-IDF series so far — only a 2-point gap between classes on both metrics.

---

# 8. ROC-AUC Analysis

| Configuration       | ROC-AUC    |
| --------------------- | ---------- |
| Baseline               | 0.8465     |
| 5K, tuned (1000 trees) | 0.8697     |
| **10K, tuned**         | **0.8714** |
| 15K, tuned             | 0.8709     |
| 20K, tuned             | 0.8687     |

ROC-AUC mirrors the accuracy trend, including the peak at 10K.

---

# 9. Cross-Validation Analysis

| Configuration       | CV Mean Accuracy | CV Std Dev |
| --------------------- | ----------------- | ---------- |
| Baseline               | 84.24%             | 0.0013     |
| 5K, tuned (180 trees)  | 85.70%             | 0.0023     |
| 5K, tuned (1000 trees) | 85.98%             | **0.0020** |
| **10K, tuned**         | **86.23%**         | 0.0029     |
| 15K, tuned             | 86.29%             | 0.0024     |
| 20K, tuned             | **86.40%**         | 0.0016     |

20K has the highest CV mean accuracy (86.40%) and the tightest std (0.0016) of the tuned runs, even though its single-split test accuracy (86.85%) is lower than 10K's (87.13%). This is a genuine case where the CV numbers and the single-split test numbers point to different "best" configurations — the CV signal is generally the more reliable one when they disagree, since it isn't sensitive to one particular test split.

---

# 10. Best Model Configuration

By single-split test accuracy, the top model was:

```python
TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

XGBClassifier(
    booster='gbtree',
    eval_metric='logloss',
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.18243763004595767,
    min_child_weight=3,
    subsample=0.7609183674204307,
    colsample_bytree=0.8525712073494107,
    random_state=0
)
```

Performance:

* Accuracy = **87.13%**
* Precision (Class 0 / 1) = **0.88 / 0.86**
* Recall (Class 0 / 1) = **0.86 / 0.88**
* ROC-AUC = **0.8714**
* Cross-Validation Accuracy = **86.23%**
* Cross-Validation Standard Deviation = **0.0029**

*(Note: by cross-validation mean, the 20K configuration is marginally stronger and more stable — Section 9. Both are legitimate candidates for "best model" depending on whether single-split test accuracy or CV mean is prioritized.)*

---

# 11. Comparison with Previously Evaluated Models (TF-IDF)

| Model                    | Best Accuracy | ROC-AUC    |
| -------------------------- | -------------- | ---------- |
| **LinearSVC**               | **90.18%**     | **0.9020** |
| Logistic Regression         | 89.46%         | 0.8948     |
| Naive Bayes (Multinomial)   | 88.13%         | 0.8815     |
| **XGBoost**                 | **87.13%**     | **0.8714** |
| Random Forest               | 85.75%         | 0.8579     |
| K-Nearest Neighbors         | 80.41%         | 0.8047     |
| Decision Tree               | 74.94%         | 0.7504     |

XGBoost outperforms Random Forest by about 1.4 points under TF-IDF, mirroring the ranking seen under CountVectorizer (where XGBoost also beat Random Forest by a similar margin). It remains behind the three linear/probabilistic models — LinearSVC, Logistic Regression, and Multinomial Naive Bayes — consistent with the broader pattern across this whole series.

---

# 12. Discussion

XGBoost's ranking relative to the other TF-IDF classifiers is consistent with what CountVectorizer showed: it comfortably beats the tree-based ensembles below it (Random Forest, Decision Tree) and KNN, but trails the linear and probabilistic models at the top of the leaderboard.

The most interesting methodological finding here is that the halving search's own resource ceiling (`max_resources=500`) undersold what the model could achieve — manually pushing `n_estimators` to 1000 (double the search's ceiling) delivered a real, if modest, improvement. This is a useful lesson for future searches on this model: consider raising `max_resources` so the search itself can find this territory, rather than relying on a manual follow-up to discover it.

The CV-vs-test-accuracy disagreement at 20K (Section 9) is also worth keeping in mind: it's a reminder that the single test-set number used to declare a "winner" throughout this series is itself only one sample, and cross-validation numbers are the more reliable signal when the two disagree.

---

# 13. Final Conclusion

This experiment evaluated XGBoost on TF-IDF features across five vocabulary configurations, using a single `HalvingRandomSearchCV` run (at 5K features) followed by a manual increase in `n_estimators` beyond the search's own resource ceiling.

The best single-split result used **TfidfVectorizer with 10,000 features and bigrams**, paired with a **gbtree XGBClassifier** (max_depth=7, learning_rate≈0.18, 1000 estimators), achieving **87.13% accuracy** and **0.8714 ROC-AUC** — placing XGBoost 4th of 7 among TF-IDF classifiers evaluated so far, ahead of Random Forest, KNN, and Decision Tree, but behind LinearSVC, Logistic Regression, and Multinomial Naive Bayes.